# Advanced - Custom properties

This notebook requires Python coding knowledge. It is also recommended to first read the notebooks [Getting started](./Getting%20started.ipynb) and [Managing properties](./Managing%20properties.ipynb).

In this notebook, we cover how to augment a model with a user-defined property by:
- defining the new property  
- implementing the way to compute it (i.e. its calculator)
- adding the new property and its associated calculator to the model  
- updating the model so the new property values are computed and stored properly for all relevant elements of the lineages.

The last part of the notebook gives some examples of custom properties and their calculators.

In [1]:
from typing import ClassVar

import numpy as np

import pycellin as pc

The data used throughout the notebook comes from the tracking of fluorescently labeled *Escherichia coli* bacteria growing on agar pad. The tracking data was obtained with TrackMate and then exported as an XML file.

In [2]:
# Path to the TrackMate XML file.
xml_path = "../sample_data/Ecoli_growth_on_agar_pad.xml"

# Parse the XML file and create a Pycellin Model object
# that contains all the data from the XML file.
model = pc.load_TrackMate_XML(xml_path)

# We can display basic information about this model.
print(model)
print(f"This model contains {model.data.number_of_lineages()} lineages:")
for lin_ID, lineage in model.data.cell_data.items():
    print(f"- ID {lin_ID}: {lineage}")

Model named 'Ecoli_growth_on_agar_pad' with 3 lineages, built from TrackMate.
This model contains 3 lineages:
- ID 0: CellLineage of ID 0 with 152 cells and 151 links.
- ID 1: CellLineage of ID 1 with 189 cells and 188 links.
- ID 2: CellLineage of ID 2 with 185 cells and 184 links.


## Definition of the new property 

A new property is defined by creating a new instance of the `Property` class, with 6 fields that describe the property:

**name**  
The name and identifier of the property. It must be **UNIQUE**: two properties cannot share the same name. This name will also be the name of the associated Python variable and must follow variable naming rules (no punctuations except underscore, no whitespaces, no numbers).

**description**  
A concise description of the property.

**provenance**  
Where does the property come from? For Pycellin computed properties, Pycellin uses `Pycellin`. For imported properties, Pycellin uses the name of the tool the data was imported from, like `TrackMate` or `CTC`. For custom properties, you can use `custom`, your initials or whatever works for you. This field is useful for traceability (e.g. reopening a model after a long time, sharing a model with other people...).

**prop_type**  
The type of graph element the property applies to, either `node`, `edge`, `lineage` or a combination of those.

**lin_type**    
Either `CellLineage`, `CycleLineage` or just `Lineage` depending on which type of lineage your property is related to. See [Pycellin data structure](./Pycellin%20data%20structure.ipynb) if in doubt.

**dtype**   
The Python type of the property values, e.g. `int`, `bool`...

**unit**    
The unit of the property values, e.g. `µm`, `min`, `cell`...


All these fields are strings. They are mandatory except `unit` which will be set to `None` if not provided.

Here is an example on a basic property, the age of the cells:

In [3]:
age_prop = pc.Property(
    identifier="age",
    name="age",
    description="Age of the cell since the start of the lineage",
    provenance="custom",
    prop_type="node",
    lin_type="CellLineage",
    dtype="int",
    unit="frame",
)

You can then access the different fields of the property:

In [4]:
print(f"{age_prop.name}: {age_prop.description} in {age_prop.unit}(s)")

age: Age of the cell since the start of the lineage in frame(s)


## Calculators

In Pycellin, a calculator is a class that defines and structures how to compute a specific property. A property cannot be associated to more than one calculator. As we will see, calculators can be very simple with just a few lines of code, or really complex. 

Currently, imported properties do not have calculators. This means that imported properties cannot be recomputed unless you define a calculator yourself and associate it with the imported property.  
Pycellin properties come with their own calculators and are stored within the `pycellin.properties` subpackage. This is a useful place to look for real examples of calculators.

When you want to compute something that is not provided by Pycellin nor imported from a tracking tool, you need to define your own calculator. Here is the stripped-down generic structure of a calculator:

``` python
class MyCalculator(CalculatorToInheritFrom):

    def compute(self, *args):
        # Code to compute the value of the property for a single element, 
        # either a node, an edge or a lineage graph.        
        return property_value
```

The calculator class must ALWAYS define at least a `compute()` method. In more complex calculators, you can also have an `__init__()`method (see the [In case of additional arguments](#in-case-of-additional-arguments) section).

With our `age` property previously defined, the calculator could be:

In [5]:
class AgeCalculator(pc.NodeGlobalPropCalculator):
    def compute(self, data: pc.Data, lineage: pc.CellLineage, nid: int) -> int:
        root = lineage.get_root()
        # FRAME is the name of the one of the time properties in TrackMate.
        return lineage.nodes[nid]["FRAME"] - lineage.nodes[root]["FRAME"]

### How to choose the calculator to inherit from?

In the example above, we have built our `AgeCalculator` by inheriting from `NodeGlobalPropCalculator`. But there are several `PropertyCalculator` base classes you can inherit from, represented in orange in the  calculators tree below. Arrows represent inheritance. 

![Calculators tree scheme](./imgs/Pycellin_calculators_scheme.png)

To know which calculator to use, you need to answer 2 questions:
1. Where do you want to store the property values? On nodes, edges, or lineage graphs?
2. What information do you need to compute your new property?

Question 1 is already answered in the previous part (not currently, but will be in next update since the property type (node, edge, lineage) is going to be one of the mandatory field of a Property). 

Question 2 requires to think about **the reach of your property**: local vs global.  
A local property is a property that only needs data from the element it will be stored in. For example, cell area would be a local property since it only needs information about the shape of the cell itself to be computed. But some properties require data from several elements or maybe from the whole lineage. In that case they are global. A simple example is our `AgeCalculator`. To compute the age of a cell you need to know when the first cell of the lineage appeared. So you need to access the data stored into 2 different cells: the cell of interest and the root of the lineage.  

These 2 examples, area and age, are examples of node properties, so their calculators would have to inherit from `NodeLocalPropCalculator` and `NodeGlobalPropCalculator` respectively.  
Computing the speed of a cell between 2 frames would need an `EdgeLocalPropCalculator` since only the position of the 2 cells forming the edge is needed.  
For lineage, mean cell speed is an example of a property based on a `LineageLocalPropCalculator`. It requires information from all the edges of the current lineage, but nothing from other lineages.

## Adding the new property and its calculator to the model

Once the property and its calculator are defined, they need to be linked together and added to the model. This is done with the `add_custom_property()` method:

``` python
# Instanciation of the calculator defined above, 
# with the property previously defined as argument.
calculator = MyCalculator(my_property)
# add_custom_property() requires an instanciated Calculator. 
model.add_custom_property(calculator)
```

For our `age` example:

In [6]:
calc = AgeCalculator(age_prop)
model.add_custom_property(calc)

Of course, it's also possible to do it in one go:

``` python
model.add_custom_property(AgeCalculator(age_prop))
```

## Updating the model

When a new property is added with `add_custom_property()`, a full update of the model is automatically planned.  

To actually run the update that will compute and add the property values for the newly added custom property, you simply need to call the `update()` method on the model:

In [7]:
model.update()

During the update, all the properties associated with a calculator will be computed or recomputed. See [Managing properties](./Managing%20properties.ipynb) for more information.

We can check that our custom `age` property has been added to all nodes by plotting a lineage and hovering over its cells:

In [8]:
lin0 = model.data.cell_data[0]
lin0.plot(
    title="Cell lineage 0 with custom `age` property",
    node_hover_props=["cell_ID", "age"],
    node_marker_style={
        "size": 10,
    },
    width=800,
    height=600,
    showlegend=False,
)

## Removing a custom property

A custom property can be removed from a model like any other properties:

In [9]:
model.remove_property("age")

See [Managing properties](./Managing%20properties.ipynb) for more information.

## Examples

### Importance of `compute()` signature

The signature of the `compute` method must be respected, even if one of the argument is not needed to compute the property.  
Here is a very simple example with the `node_ID_parity` property:

In [10]:
prop_incorrect = pc.Property(
    identifier="node_ID_parity_incorrect",
    name="node ID parity_incorrect",
    description="Parity of the node ID",
    provenance="Pycellin",
    prop_type="node",
    lin_type="CellLineage",
    dtype="int",
)


class ParityCalculator_incorrect(pc.NodeLocalPropCalculator):
    def compute(self, nid):
        return nid % 2 == 0

In [11]:
calc = ParityCalculator_incorrect(prop_incorrect)
model.add_custom_property(calc)
model.update()

TypeError: ParityCalculator_incorrect.compute() takes 2 positional arguments but 3 were given

The update cannot be done because the custom calculator does not abide by `compute()` signature: it is missing the `lineage` argument. The signature is enforced to provide a relatively simple but consistent API. The arguments expected by `compute()` depend on the inherited calculator, i.e. both on the property type (node, edge or lineage) and the property reach (local or global). 

Here is the correct way to define the `node_ID_parity` property:

In [12]:
prop_correct = pc.Property(
    identifier="node_ID_parity_correct",
    name="node ID parity_correct",
    description="Parity of the node ID",
    provenance="Pycellin",
    prop_type="node",
    lin_type="CellLineage",
    dtype="int",
)


class ParityCalculator_correct(pc.NodeLocalPropCalculator):
    def compute(self, lineage, nid):
        # `lineage` is unused here, but it is still a required argument.
        return nid % 2 == 0

In [13]:
calc = ParityCalculator_correct(prop_correct)
model.add_custom_property(calc)

# We clean the model by removing the incorrect property,
# otherwise we won't be able to update the model.
model.remove_property("node_ID_parity_incorrect")
model.update()

# Checking that the property was correctly added.
print(lin0.nodes[8993]["node_ID_parity_correct"])
print(lin0.nodes[9014]["node_ID_parity_correct"])

False
True


### In case of additional arguments

Sometimes, your custom property needs additional arguments to be computed, for example specific information from the model, like pixel size or the time elapsed between frames.  
In that case, you need to overwrite the `__init__` method of the `PropertyCalculator` parent class. Here is a snippet of this class:

``` python
# Code from pycellin.classes.property_calculator.py

from abc import ABC

class PropertyCalculator(ABC):
    """
    Abstract class to compute property values.
    """

    def __init__(self, property: pc.Property):
        self.property = property
```

So the `__init__` of your calculator must:
1. take your new `Property` as argument;
2. call the parent calculator own `__init__` on the `Property` with `super().__init__(property)`.

Below, a concrete example on the relative age of a cell, given in time units:

In [14]:
age_prop2 = pc.Property(
    identifier="relative_age",
    name="relative age",
    description="Age of the cell since the start of the current cell cycle",
    provenance="custom",
    prop_type="node",
    lin_type="CellLineage",
    dtype="float",
    unit=model.get_time_unit(),
)


class MyRelativeAgeCalculator(pc.NodeGlobalPropCalculator):
    def __init__(self, property: pc.Property, time_prop_name: str):
        # Call the parent __init__ method.
        super().__init__(property)
        # Do whatever you want with the additional argument(s).
        # Usually you store them in the object to be able to use them
        # in the `compute` method.
        self.time_prop_name = time_prop_name

    def compute(self, data: pc.Data, lineage: pc.CellLineage, nid: int):
        # As said before, the signature of the `compute` method must be respected,
        # even if we don't use the `data` argument here.
        first_cell = lineage.get_cell_cycle(nid)[0]
        # We use the additional time_prop_name argument so that this method can work
        # with any time property, including the reference time property of the model.
        first_cell_time = lineage.nodes[first_cell][self.time_prop_name]
        current_cell_time = lineage.nodes[nid][self.time_prop_name]
        return current_cell_time - first_cell_time

After that, the property can be added to the model as usual.

In [15]:
calc = MyRelativeAgeCalculator(age_prop2, model.reference_time_property)
model.add_custom_property(calc)

In [16]:
model.update()
lin0.plot(
    title="Cell lineage 0 with custom `relative_age` property",
    node_hover_props=["cell_ID", "relative_age"],
    node_marker_style={
        "size": 10,
    },
    width=800,
    height=600,
    showlegend=False,
)

### Declaring the properties your calculator reads

`MyRelativeAgeCalculator` above reads a property whose identifier is chosen by the user (`time_prop_name`). If that identifier is wrong, nothing tells you until `update()` fails. To have it checked when the property is added, like for Pycellin properties (see [Managing properties](./Managing%20properties.ipynb)):
1. store the identifier of each input property in an attribute of the calculator;
2. list these attributes in the `INPUT_PROPS` class attribute, with the property type and lineage type each input property must have.

`add_custom_property()` then checks that each input property is declared in the model, with the right types. For `MyRelativeAgeCalculator`, that would be `INPUT_PROPS = {"time_prop_name": ("node", "CellLineage")}`.

The shape factor below is computed from an area and a perimeter: it is 1 for a disk, and larger for elongated cells. By default, it reads Pycellin's `cell_area` and `cell_perimeter`.

The `ClassVar[...]` annotation is optional: without it, everything works the same, but linters such as Ruff warn that a dictionary defined on the class could be modified by mistake.

In [17]:
class ShapeFactor(pc.NodeLocalPropCalculator):
    INPUT_PROPS: ClassVar[dict[str, tuple[str, str]]] = {
        "area_prop": ("node", "CellLineage"),
        "perimeter_prop": ("node", "CellLineage"),
    }

    def __init__(
        self,
        property: pc.Property,
        area_prop: str = "cell_area",
        perimeter_prop: str = "cell_perimeter",
    ):
        super().__init__(property)
        self.area_prop = area_prop
        self.perimeter_prop = perimeter_prop

    def compute(self, lineage: pc.CellLineage, nid: int) -> float:
        cell = lineage.nodes[nid]
        return cell[self.perimeter_prop] ** 2 / (4 * np.pi * cell[self.area_prop])


shape_factor_prop = pc.Property(
    identifier="shape_factor",
    name="shape factor",
    description="Perimeter squared divided by 4 pi times the area",
    provenance="custom",
    prop_type="node",
    lin_type="CellLineage",
    dtype="float",
)

Pycellin's `cell_area` and `cell_perimeter` are not in this model, so adding the property with the default inputs fails right away, and the error says how to fix it:

In [18]:
model.add_custom_property(ShapeFactor(shape_factor_prop))

MissingPropertyError: 'shape_factor' requires the property 'cell_area', which has not been declared. To fix this, add it first with model.add_cell_area(), or set 'area_prop' to the identifier of a declared property.

TrackMate computed the area and perimeter of each cell (`AREA` and `PERIMETER`), so we can use them instead:

In [19]:
model.add_custom_property(
    ShapeFactor(shape_factor_prop, area_prop="AREA", perimeter_prop="PERIMETER")
)
model.update()
model.to_cell_dataframe()[["cell_ID", "AREA", "PERIMETER", "shape_factor"]].head()

,cell_ID,AREA,PERIMETER,shape_factor
0,8993,2.666228,10.285699,3.157624
1,9013,1.282132,5.944787,2.193461
2,9014,1.262608,5.747940,2.082314
3,8986,1.462195,6.355476,2.198270
4,8989,1.227897,6.341672,2.606369


### Growth rate: a cycle lineage property

In [20]:
gr_prop = pc.Property(
    identifier="growth_rate",
    name="growth rate",
    description=(
        "Growth rate of the bacterium during a cell cycle, "
        "i.e. (area_at_div_time - area_at_birth_time) / division_time"
    ),
    provenance="custom",
    prop_type="node",
    lin_type="CycleLineage",
    dtype="float",
    unit=f"{model.get_space_unit()}2/{model.get_time_unit()}",
)


class GrowthRateCalculator(pc.NodeGlobalPropCalculator):
    def compute(self, data, lineage, nid):
        cell_lin = data.cell_data[lineage.graph["lineage_ID"]]
        birth_cell = lineage.nodes[nid]["cells"][0]
        area_birth = cell_lin.nodes[birth_cell]["AREA"]
        div_cell = lineage.nodes[nid]["cells"][-1]
        area_div = cell_lin.nodes[div_cell]["AREA"]
        div_time = lineage.nodes[nid]["division_time"]
        if div_time == 0 or area_div - area_birth == 0:
            return np.nan
        else:
            return (area_div - area_birth) / div_time

In [21]:
model.add_cycle_data()
# Division time is a Pycellin property, and it is needed to compute the growth rate,
# see line 23 in the code above.
model.add_division_time()
model.add_custom_property(GrowthRateCalculator(gr_prop))
model.update()
model.data.cycle_data[0].plot(
    title="Cycle lineage 0 with custom `growth_rate` property",
    node_hover_props=["cycle_ID", "growth_rate"],
    node_marker_style={
        "size": 10,
    },
    width=800,
    height=300,
    showlegend=False,
)